# Sentiment Analysis - A100 Optimized
## Metacritic Game Reviews (Colab A100 Version)
**GPU**: NVIDIA A100 40GB/80GB with Mixed Precision Training

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

MODELS_DIR = '/content/drive/MyDrive/sentiment_analyses_colab/models'
DATA_DIR = '/content/drive/MyDrive/sentiment_analyses_colab/data_all'

In [ ]:
import warnings
import multiprocessing
warnings.filterwarnings("ignore", message="can only test a child process")
import os, pickle, joblib, re
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler  # Mixed Precision
from transformers import DistilBertTokenizer, DistilBertModel, get_linear_schedule_with_warmup
import shap
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# A100 Optimized Settings
RANDOM_STATE = 42
LSTM_BATCH_SIZE = 1024      # A100: 256 vs T4: 64
BERT_BATCH_SIZE = 256       # A100: 64 vs T4: 16
BERT_SAMPLE_SIZE = 150000   # A100: 50K+ vs T4: 10K
HIDDEN_DIM = 256           # A100: 256 vs T4: 128
EMBED_DIM = 256            # A100: 256 vs T4: 128
NUM_LAYERS = 3             # A100: 3 vs T4: 2
EPOCHS = 20                # A100: 10 vs T4: 5
MAX_LEN = 200              # A100: 200 vs T4: 150

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# A100 specific optimizations
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

os.makedirs(MODELS_DIR, exist_ok=True)
print(f'Device: {DEVICE}')

## 1. Load Data

In [ ]:
print('Loading data...')
X_train = np.load(f'{DATA_DIR}/X_train.npy', allow_pickle=True)
X_test = np.load(f'{DATA_DIR}/X_test.npy', allow_pickle=True)
y_train = np.load(f'{DATA_DIR}/y_train.npy')
y_test = np.load(f'{DATA_DIR}/y_test.npy')
X_train_tfidf = sparse.load_npz(f'{DATA_DIR}/X_train_tfidf.npz')
X_test_tfidf = sparse.load_npz(f'{DATA_DIR}/X_test_tfidf.npz')
X_train_svd = np.load(f'{DATA_DIR}/X_train_svd.npy')
X_test_svd = np.load(f'{DATA_DIR}/X_test_svd.npy')

with open(f'{DATA_DIR}/tfidf_vectorizer.pkl', 'rb') as f: tfidf_vectorizer = pickle.load(f)
with open(f'{DATA_DIR}/svd_model.pkl', 'rb') as f: svd = pickle.load(f)
with open(f'{DATA_DIR}/vocab.pkl', 'rb') as f: vocab = pickle.load(f)
with open(f'{DATA_DIR}/stopwords.pkl', 'rb') as f: STOPWORDS = pickle.load(f)
vocab_size = len(vocab) + 1

print(f'Train: {len(X_train):,}, Test: {len(X_test):,}, Vocab: {vocab_size:,}')

## 2. Ridge & XGBoost

In [ ]:
# Ridge
print('Training Ridge...')
ridge_model = GridSearchCV(Ridge(), {'alpha': [0.1, 1.0, 10.0, 100.0]}, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
ridge_model.fit(X_train_tfidf, y_train)
y_pred_ridge = np.clip(ridge_model.predict(X_test_tfidf), 0, 100)
ridge_mae, ridge_mse, ridge_r2 = mean_absolute_error(y_test, y_pred_ridge), mean_squared_error(y_test, y_pred_ridge), r2_score(y_test, y_pred_ridge)
print(f'Ridge - MAE: {ridge_mae:.4f}, R2: {ridge_r2:.4f}')
joblib.dump(ridge_model.best_estimator_, f'{MODELS_DIR}/ridge_model.pkl')

# XGBoost with GPU
print('\nTraining XGBoost (GPU)...')
xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8,
    tree_method='hist', device='cuda',  # Changed 'gpu_hist' to 'hist'
    random_state=RANDOM_STATE, early_stopping_rounds=20
)
xgb_model.fit(X_train_svd, y_train, eval_set=[(X_test_svd, y_test)], verbose=False)
y_pred_xgb = np.clip(xgb_model.predict(X_test_svd), 0, 100)
xgb_mae, xgb_mse, xgb_r2 = mean_absolute_error(y_test, y_pred_xgb), mean_squared_error(y_test, y_pred_xgb), r2_score(y_test, y_pred_xgb)
print(f'XGBoost - MAE: {xgb_mae:.4f}, R2: {xgb_r2:.4f}')
xgb_model.save_model(f'{MODELS_DIR}/xgboost_model.json')

## 3. Bi-LSTM with Attention (A100 Optimized)

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, scores, vocab, max_len):
        self.texts, self.scores, self.vocab, self.max_len = texts, scores, vocab, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        tokens = self.texts[idx].split()[:self.max_len]
        indices = [self.vocab.get(t, 0) for t in tokens] + [0] * (self.max_len - len(tokens))
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.scores[idx], dtype=torch.float)

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, x, mask=None):
        scores = self.attention(x).squeeze(-1)
        if mask is not None: scores = scores.masked_fill(mask == 0, -1e4)
        weights = F.softmax(scores, dim=1)
        return torch.bmm(weights.unsqueeze(1), x).squeeze(1), weights

class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, bidirectional=True, batch_first=True, dropout=dropout)
        self.attention = Attention(hidden_dim * 2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),  # Better than ReLU
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        mask = (x != 0).float()
        emb = self.dropout(self.layer_norm(self.embedding(x)))
        out, _ = self.lstm(emb)
        ctx, self.attn_weights = self.attention(out, mask)
        return self.fc(ctx).squeeze(1)

print(f'Model Config: embed={EMBED_DIM}, hidden={HIDDEN_DIM}, layers={NUM_LAYERS}')

In [ ]:
# DataLoaders with larger batch size
train_loader = DataLoader(ReviewDataset(X_train, y_train, vocab, MAX_LEN), batch_size=LSTM_BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(ReviewDataset(X_test, y_test, vocab, MAX_LEN), batch_size=LSTM_BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

lstm_model = BiLSTMAttention(vocab_size).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.002, epochs=EPOCHS, steps_per_epoch=len(train_loader))
scaler = GradScaler()  # Mixed Precision

print(f'Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')
print(f'Batch Size: {LSTM_BATCH_SIZE}, Epochs: {EPOCHS}')

best_loss = float('inf')
for epoch in range(EPOCHS):
    lstm_model.train()
    total_loss = 0
    for batch_x, batch_y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        batch_x, batch_y = batch_x.to(DEVICE, non_blocking=True), batch_y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()

        # Mixed Precision Training
        with autocast():
            outputs = lstm_model(batch_x)
            loss = criterion(outputs, batch_y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1} - Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}')
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(lstm_model.state_dict(), f'{MODELS_DIR}/bilstm_attention_best.pt')

print(f'[OK] Best Loss: {best_loss:.4f}')

In [ ]:
lstm_model.eval()
y_pred_lstm = []
with torch.no_grad():
    for batch_x, _ in tqdm(test_loader, desc='Evaluating'):
        with autocast():
            outputs = lstm_model(batch_x.to(DEVICE))
        y_pred_lstm.extend(outputs.cpu().numpy())
y_pred_lstm = np.clip(np.array(y_pred_lstm), 0, 100)
lstm_mae, lstm_mse, lstm_r2 = mean_absolute_error(y_test, y_pred_lstm), mean_squared_error(y_test, y_pred_lstm), r2_score(y_test, y_pred_lstm)
print(f'Bi-LSTM - MAE: {lstm_mae:.4f}, MSE: {lstm_mse:.4f}, R2: {lstm_r2:.4f}')

## 4. DistilBERT (A100 - Full Power)

In [ ]:
class BertDataset(Dataset):
    def __init__(self, texts, scores, tokenizer, max_len=128):
        self.texts, self.scores, self.tokenizer, self.max_len = texts, scores / 100.0, tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze(), 'score': torch.tensor(self.scores[idx], dtype=torch.float)}

class DistilBertRegressor(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.regressor = nn.Sequential(
            nn.Linear(768, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.regressor(out.last_hidden_state[:, 0, :]).squeeze(1)

# Use more data with A100
BERT_SAMPLE = min(BERT_SAMPLE_SIZE, len(X_train))
bert_idx = np.random.choice(len(X_train), BERT_SAMPLE, replace=False)
X_train_bert, y_train_bert = X_train[bert_idx], y_train[bert_idx]
bert_test_idx = np.random.choice(len(X_test), min(5000, len(X_test)), replace=False)
X_test_bert, y_test_bert = X_test[bert_test_idx], y_test[bert_test_idx]

print(f'BERT Train: {len(X_train_bert):,}, Test: {len(X_test_bert):,}')

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_train_loader = DataLoader(BertDataset(X_train_bert, y_train_bert, tokenizer), batch_size=BERT_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
bert_test_loader = DataLoader(BertDataset(X_test_bert, y_test_bert, tokenizer), batch_size=BERT_BATCH_SIZE, num_workers=4, pin_memory=True)

bert_model = DistilBertRegressor().to(DEVICE)
bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=3e-5, weight_decay=0.01)
bert_scheduler = get_linear_schedule_with_warmup(bert_optimizer, num_warmup_steps=len(bert_train_loader), num_training_steps=len(bert_train_loader)*5)
bert_scaler = GradScaler()

print(f'BERT Batch: {BERT_BATCH_SIZE}, Steps/Epoch: {len(bert_train_loader)}')

best_bert = float('inf')
for epoch in range(10):
    bert_model.train()
    total_loss = 0
    for batch in tqdm(bert_train_loader, desc=f'BERT {epoch+1}/10'):
        bert_optimizer.zero_grad()
        with autocast():
            out = bert_model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            loss = F.mse_loss(out, batch['score'].to(DEVICE))
        bert_scaler.scale(loss).backward()
        bert_scaler.unscale_(bert_optimizer)
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        bert_scaler.step(bert_optimizer)
        bert_scaler.update()
        bert_scheduler.step()
        total_loss += loss.item()
    avg = total_loss / len(bert_train_loader)
    print(f'Epoch {epoch+1} Loss: {avg:.4f}')
    if avg < best_bert:
        best_bert = avg
        torch.save(bert_model.state_dict(), f'{MODELS_DIR}/distilbert_best.pt')

## RE-Upload the Modals

In [ ]:
ridge_model = joblib.load(f'{MODELS_DIR}/ridge_model.pkl')

xgb_model = xgb.XGBRegressor()
xgb_model.load_model(f'{MODELS_DIR}/xgboost_model.json')

# Re-instantiate tokenizer and bert_model before loading state_dict
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertRegressor().to(DEVICE)
bert_model.load_state_dict(torch.load(f'{MODELS_DIR}/distilbert_best.pt', map_location=DEVICE))
bert_model.eval()

# Re-instantiate lstm_model before loading state_dict
lstm_model = BiLSTMAttention(vocab_size).to(DEVICE)
lstm_model.load_state_dict(torch.load(f'{MODELS_DIR}/bilstm_attention_best.pt', map_location=DEVICE))
lstm_model.eval()

print('Models Uploaded')

In [ ]:
bert_model.eval()
y_pred_bert = []
with torch.no_grad():
    for batch in tqdm(bert_test_loader, desc='BERT Eval'):
        with autocast():
            out = bert_model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        y_pred_bert.extend((out.cpu().numpy() * 100))
y_pred_bert = np.clip(np.array(y_pred_bert), 0, 100)
bert_mae, bert_mse, bert_r2 = mean_absolute_error(y_test_bert, y_pred_bert), mean_squared_error(y_test_bert, y_pred_bert), r2_score(y_test_bert, y_pred_bert)
print(f'DistilBERT - MAE: {bert_mae:.4f}, MSE: {bert_mse:.4f}, R2: {bert_r2:.4f}')

## 5. Results

In [ ]:
results = pd.DataFrame({
    'Model': ['Ridge', 'XGBoost', 'Bi-LSTM+Attn', 'DistilBERT'],
    'MAE': [ridge_mae, xgb_mae, lstm_mae, bert_mae],
    'MSE': [ridge_mse, xgb_mse, lstm_mse, bert_mse],
    'R2': [ridge_r2, xgb_r2, lstm_r2, bert_r2]
})
print('='*50)
print('A100 PERFORMANCE RESULTS')
print('='*50)
print(results.to_string(index=False))
results.to_csv(f'{MODELS_DIR}/results_a100.csv', index=False)

# Plot
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for i, m in enumerate(['MAE', 'MSE', 'R2']):
    bars = ax[i].bar(results['Model'], results[m], color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6'])
    ax[i].set_title(m, fontweight='bold')
    ax[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/results_a100.png', dpi=150)
plt.show()

## 6. Model Explainability

SHAP for XGBoost:

In [ ]:
# SHAP Explainability for XGBoost
print("Generating SHAP explanations for XGBoost...")

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_svd[:1000])

# Summary Plot
plt.figure(figsize=(12, 6))
shap.summary_plot(shap_values, X_test_svd[:1000], show=False)
plt.title("XGBoost Feature Importance (SHAP)")
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/shap_summary.png', dpi=150)
plt.show()

Attention Visualization:

In [ ]:
# Bi-LSTM Attention Visualization
def visualize_attention(text, model, vocab, max_len=MAX_LEN):
    """Visualize which words the model focuses on"""
    clean = re.sub(r'[^a-z0-9\s]', '', text.lower())
    tokens = clean.split()[:max_len]

    indices = [vocab.get(t, 0) for t in tokens] + [0] * (max_len - len(tokens))
    inp = torch.tensor([indices], dtype=torch.long).to(DEVICE)

    model.eval()
    with torch.no_grad():
        pred = model(inp).item()
        weights = model.attn_weights[0, :len(tokens)].cpu().numpy()

    # Normalize weights
    weights = weights / weights.max()

    # Create visualization
    fig, ax = plt.subplots(figsize=(14, 3))

    # Color map based on attention weights
    colors = plt.cm.Reds(weights)

    for i, (word, weight) in enumerate(zip(tokens, weights)):
        ax.text(i, 0, word, fontsize=12, ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor=colors[i], alpha=0.8))

    ax.set_xlim(-0.5, len(tokens) - 0.5)
    ax.set_ylim(-0.5, 0.5)
    ax.axis('off')
    ax.set_title(f'Attention Weights (Predicted Score: {pred:.1f})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{MODELS_DIR}/attention_viz.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print top attended words
    word_weights = list(zip(tokens, weights))
    word_weights.sort(key=lambda x: x[1], reverse=True)
    print("\nTop Attended Words:")
    for word, weight in word_weights[:10]:
        bar = '█' * int(weight * 20)
        print(f"  {word:15s} {bar} ({weight:.3f})")

# Examples
print("="*60)
print("ATTENTION VISUALIZATION")
print("="*60)

visualize_attention("This game is absolutely amazing with great graphics and gameplay!", lstm_model, vocab)
visualize_attention("Terrible performance, constant bugs and crashes. Waste of money.", lstm_model, vocab)
visualize_attention("Good game but has some issues. Worth playing if on sale.", lstm_model, vocab)

Word Importance Analysis:

In [ ]:
# Word-level importance analysis
def analyze_word_importance(text, model, vocab, max_len=MAX_LEN):
    """Analyze how each word affects the prediction"""
    clean = re.sub(r'[^a-z0-9\s]', '', text.lower())
    tokens = clean.split()[:max_len]

    # Base prediction
    indices = [vocab.get(t, 0) for t in tokens] + [0] * (max_len - len(tokens))
    inp = torch.tensor([indices], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        base_pred = model(inp).item()

    # Remove each word and see impact
    impacts = []
    for i in range(len(tokens)):
        masked = tokens[:i] + tokens[i+1:]
        indices = [vocab.get(t, 0) for t in masked] + [0] * (max_len - len(masked))
        inp = torch.tensor([indices], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            new_pred = model(inp).item()
        impact = base_pred - new_pred  # Positive = word increases score
        impacts.append((tokens[i], impact))

    # Sort by absolute impact
    impacts.sort(key=lambda x: abs(x[1]), reverse=True)

    print(f"Base Prediction: {base_pred:.1f}")
    print("\nWord Impact Analysis:")
    for word, impact in impacts[:10]:
        direction = "↑" if impact > 0 else "↓"
        bar = '█' * int(abs(impact) * 2)
        print(f"  {word:15s} {direction} {bar} ({impact:+.2f})")

    return impacts

print("\n" + "="*60)
print("WORD IMPACT ANALYSIS")
print("="*60)
analyze_word_importance("Amazing graphics but terrible gameplay and constant crashes", lstm_model, vocab)

In [ ]:
# Interactive prediction
def predict(text):
    clean = re.sub(r'[^a-z0-9\s]', '', text.lower())
    tfidf = tfidf_vectorizer.transform([clean])
    r = ridge_model.predict(tfidf)[0] # Changed from ridge_model.best_estimator_.predict(tfidf)[0]
    x = xgb_model.predict(svd.transform(tfidf))[0]

    if(r < 0): r = 0
    if(x < 0): x = 0

    tokens = clean.split()[:MAX_LEN]
    idx = [vocab.get(t, 0) for t in tokens] + [0] * (MAX_LEN - len(tokens))
    inp = torch.tensor([idx], dtype=torch.long).to(DEVICE)
    with torch.no_grad(), autocast():
        l = lstm_model(inp).item()

    enc = tokenizer(clean, truncation=True, max_length=128, padding='max_length', return_tensors='pt')
    with torch.no_grad(), autocast():
        b = bert_model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE)).item() * 100

    print(f'Ridge: {r:.1f} | XGB: {x:.1f} | LSTM: {l:.1f} | BERT: {b:.1f} | Avg: {np.mean([r,x,l,b]):.1f}')

predict("Amazing game with amazing graphics!")
predict("Terrible bugs, worst game ever.")